# Embed `chunks.json` bằng `bge-m3` trên Colab

Notebook **tự chứa** — không cần clone repo, không import `src/`.
Chỉ cần đưa lên Colab **2 thứ**: file này và **một** file `*.chunks.json`.

```
VÀO   out/kb/<ten>.chunks.json          (ChunkSet — sinh bởi src/kb/cli.py)
RA    <doc_id>__bge-m3.vectors.npy      ma trận (N, 1024) float32, đã chuẩn hoá L2
      <doc_id>__bge-m3.vectors.json     {model, dim, pooling, normalized, rows:[chunk_id...]}
      <doc_id>__bge-m3.sparse.json      {chunk_id: {token_id: weight}}   ← nhánh từ khoá
      audit/self_retrieval.json         kết quả gate §11 (top-1 ≥ 90%)
```

Bốn luật lấy từ `docs/spec/embedding.md`, notebook này tuân đúng:

| # | Luật | Chỗ thực thi |
|---|---|---|
| 1 | Embed `text_enriched`, **KHÔNG** `text_raw` (CLAUDE.md §10) | cell *Nhúng cả bộ* |
| 2 | `bge-m3` gộp bằng **CLS**, không phải mean — sai thì **không báo lỗi**, chỉ tụt chất lượng | `encode()` |
| 3 | Nhúng **CẢ** N chunk, kể cả trang phân mục. Lọc là việc của lúc TRUY VẤN | cell *Nhúng cả bộ* / `search()` |
| 4 | `model_id` nằm trong **TÊN FILE** | cell *Ghi file* |

**Trước khi chạy:** `Runtime → Change runtime type → GPU (T4)`.
Chạy được cả trên CPU, chỉ là chậm hơn.

> Cảnh báo về số đo: latency đo trên Colab **không** trả lời được câu hỏi của spec §2
> (ngân sách `< 2.5s` tính trên máy chạy runtime, CPU 8 luồng). Colab GPU nhanh hơn,
> Colab CPU (2 vCPU) chậm hơn. Xem cell *Đo tốc độ* để biết đọc số nào.

## 1. Môi trường

Nếu Colab bảo *restart runtime* sau cell này: `Runtime → Restart session`, rồi chạy lại
**từ cell này** (file đã upload vẫn nằm trong `/content`, không mất).

In [ ]:
%pip -q install -U "transformers>=4.44" "huggingface_hub>=0.25" sentencepiece

import platform, torch, transformers
print("python      ", platform.python_version())
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(chay CPU)")

## 2. Nạp `chunks.json`

Tự tìm file trong `/content` trước; không thấy thì mở hộp thoại upload.

In [ ]:
import glob, json
from pathlib import Path
from collections import Counter
import statistics

cands = [p for p in sorted(glob.glob("/content/**/*chunks*.json", recursive=True))
         if "/sample_data/" not in p]
if not cands:
    from google.colab import files
    up = files.upload()                       # chon file *.chunks.json
    cands = [f"/content/{n}" for n in up]

CHUNKS_PATH = Path(cands[0])
CS = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
chunks = CS["chunks"]

need = {"chunk_id", "page_no", "text_raw", "text_enriched", "vector_role", "content_type"}
missing = need - set(chunks[0])
assert not missing, f"chunks.json thieu truong: {missing}"

tok_counts = [c.get("token_count", 0) for c in chunks]
print("file       ", CHUNKS_PATH.name)
print("doc_id     ", CS["doc_id"])
print("tokenizer  ", CS.get("tokenizer", "(khong ghi)"))
print("chunk      ", len(chunks))
print("vai tro    ", dict(Counter(c["vector_role"] for c in chunks)))
print("loai       ", dict(Counter(c["content_type"] for c in chunks)))
print("token      min=%d trung vi=%.0f max=%d" %
        (min(tok_counts), statistics.median(tok_counts), max(tok_counts)))
print("trang      ", min(c["page_no"] for c in chunks), "->",
      max(c["page_no"] for c in chunks))
over = [c["chunk_id"] for c in chunks if c.get("token_count", 0) > CS.get("max_tokens", 500)]
print("vuot max_tokens:", over or "khong co")

## 3. Cấu hình

Giữ **y hệt** `src/kb/embed.py` để vector sinh ở đây và vector sinh ở máy là một thứ.
Đổi bất kỳ dòng nào dưới đây (nhất là `POOLING`, `MAX_LEN`, `NORMALIZE`) là phải sinh lại
toàn bộ, không được trộn.

In [ ]:
import torch

MODEL_ID    = "BAAI/bge-m3"
POOLING     = "cls"        # spec §5 - bge-m3 lay vector o token CLS, KHONG phai mean
MAX_LEN     = 512          # chunk to nhat do duoc 353 token
NORMALIZE   = True         # spec §6 - L2 -> cosine = tich vo huong
BATCH       = 16           # GPU thoai mai; CPU nen ha ve 8
USE_FP16    = False        # False = trung so voi ban chay CPU o may; True = nhanh hon chut
WANT_SPARSE = True         # spec §9 - nhanh tu khoa cua hybrid
SPARSE_MIN  = 0.01         # bo trong so vun cho file nho lai

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SLUG = MODEL_ID.split("/")[-1].lower()
OUT        = Path("/content/out/kb")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "audit").mkdir(exist_ok=True)

print("device", DEVICE, "| fp16", USE_FP16, "| sparse", WANT_SPARSE, "| out", OUT)

## 4. Nạp model

~2.2 GB tải một lần. `AutoModel` **không** đụng tới thư mục `onnx` trong repo (đúng ý
spec §2: bỏ `onnx`, nó là bản sao cùng model, thêm 2.2 GB mà không dùng tới).

In [ ]:
import time
from transformers import AutoModel, AutoTokenizer

t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16 if USE_FP16 else torch.float32
).to(DEVICE).eval()
LOAD_SEC = time.perf_counter() - t0

DIM = int(model.config.hidden_size)
print("nap %.1fs | %.0fM tham so | %d chieu" %
        (LOAD_SEC, sum(p.numel() for p in model.parameters()) / 1e6, DIM))

sparse_linear = None
if WANT_SPARSE:
    from huggingface_hub import hf_hub_download
    sd = torch.load(hf_hub_download(MODEL_ID, "sparse_linear.pt"), map_location="cpu")
    sparse_linear = torch.nn.Linear(DIM, 1)
    sparse_linear.load_state_dict(sd)
    sparse_linear = sparse_linear.to(DEVICE).float().eval()
    print("sparse_linear: da nap ->", tuple(sparse_linear.weight.shape))

## 5. Hàm nhúng

Hai chỗ dễ sai im lặng, đều nằm trong cell này:

* **CLS, không mean.** `hid[:, 0]` chứ không phải trung bình theo `attention_mask`.
  Dùng mean thì vector vẫn đủ 1024 chiều, code vẫn chạy, không có exception nào —
  chỉ chất lượng truy xuất tụt mà không ai truy ra nguyên nhân.
* **Sparse phải bỏ special token.** `<s>`, `</s>`, `<pad>`, `<unk>` bị loại trước khi gộp;
  token trùng nhau thì lấy **max**, không cộng dồn.

In [ ]:
import numpy as np

SPECIAL_IDS = set(tok.all_special_ids)


def _sparse_rows(hid, input_ids, attn):
    w = torch.relu(sparse_linear(hid.float())).squeeze(-1) * attn      # (B, L)
    rows = []
    for ids, ws in zip(input_ids.tolist(), w.float().cpu().tolist()):
        d = {}
        for i, t in zip(ids, ws):
            if i in SPECIAL_IDS or t <= SPARSE_MIN:
                continue
            if t > d.get(i, 0.0):                                      # trung token -> max
                d[i] = t
        rows.append({str(k): round(v, 4)
                     for k, v in sorted(d.items(), key=lambda kv: -kv[1])})
    return rows


@torch.no_grad()
def encode(texts, batch=BATCH, want_sparse=False, log_every=0):
    dense, sparse = [], []
    for s in range(0, len(texts), batch):
        enc = tok(texts[s:s + batch], padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        hid = model(**enc).last_hidden_state
        if POOLING == "cls":
            v = hid[:, 0]
        else:
            m = enc["attention_mask"].unsqueeze(-1).float()
            v = (hid * m).sum(1) / m.sum(1)
        if NORMALIZE:
            v = torch.nn.functional.normalize(v.float(), dim=1)
        dense.append(v.float().cpu().numpy())
        if want_sparse and sparse_linear is not None:
            sparse.extend(_sparse_rows(hid, enc["input_ids"], enc["attention_mask"]))
        if log_every and (s // batch) % log_every == 0:
            print("  %d/%d" % (min(s + batch, len(texts)), len(texts)))
    return np.vstack(dense).astype(np.float32), sparse

## 6. Nhúng cả bộ

`text_enriched`, tất cả N chunk, **không lọc gì cả**.

Tiền tố `[<muc> · trang k/N]` phải nằm trong vector: deck này có 7 trang liền nhau cùng
tiêu đề, tiền tố là thứ duy nhất tách được chúng — cell *Kiểm tra trang trùng tiêu đề* ở
dưới đo xem tách được bao nhiêu.

In [ ]:
texts = [c["text_enriched"] for c in chunks]         # §10 CAM dung text_raw

t0 = time.perf_counter()
MAT, SPARSE = encode(texts, want_sparse=WANT_SPARSE, log_every=1)
if DEVICE == "cuda":
    torch.cuda.synchronize()
EMBED_SEC = time.perf_counter() - t0

print("nhung %d vector trong %.1fs (%.0f ms/chunk) -> %s" %
        (len(texts), EMBED_SEC, EMBED_SEC / len(texts) * 1000, MAT.shape))
print("chuan hoa L2: norm min=%.4f max=%.4f (phai ~1.0)" %
        (np.linalg.norm(MAT, axis=1).min(), np.linalg.norm(MAT, axis=1).max()))
if SPARSE:
    print("sparse: trung binh %.0f token/chunk" % (sum(len(r) for r in SPARSE) / len(SPARSE)))

## 7. Ghi file

`model_id` **nằm trong tên file** — luật §5 S6a của CLAUDE.md áp cho file thay vì Qdrant
collection. Đổi model mà quên sinh lại thì runtime truy vấn index cũ bằng vector mới, trả
rác mà **không báo lỗi**; tên file khác nhau thì không thể vô tình dùng nhầm.

Tên file lấy nguyên `doc_id` (kể cả khoảng trắng) cho khớp đúng `src/kb/embed.py`.

In [ ]:
stem     = f'{CS["doc_id"]}__{MODEL_SLUG}'
p_npy    = OUT / f"{stem}.vectors.npy"
p_json   = OUT / f"{stem}.vectors.json"
p_sparse = OUT / f"{stem}.sparse.json"

np.save(p_npy, MAT)
if SPARSE:
    p_sparse.write_text(
        json.dumps({c["chunk_id"]: w for c, w in zip(chunks, SPARSE)}, ensure_ascii=False),
        encoding="utf-8")

p_json.write_text(json.dumps({
    "doc_id": CS["doc_id"],
    "model": MODEL_ID,
    "dim": int(MAT.shape[1]),
    "pooling": POOLING,
    "normalized": NORMALIZE,
    "max_len": MAX_LEN,
    "n_vectors": int(MAT.shape[0]),
    "embed_sec": round(EMBED_SEC, 2),
    "device": DEVICE,
    "fp16": USE_FP16,
    "sparse": bool(SPARSE),
    "sparse_file": p_sparse.name if SPARSE else None,
    "sparse_min_weight": SPARSE_MIN if SPARSE else None,
    "source_chunks": CHUNKS_PATH.name,
    "rows": [c["chunk_id"] for c in chunks],   # .npy chi la ma tran - hang nao la chunk nao o day
}, ensure_ascii=False, indent=2), encoding="utf-8")

for p in (p_npy, p_json, p_sparse):
    if p.exists():
        print("%8d KB  %s" % (p.stat().st_size // 1024, p.name))

## 8. Đo tốc độ

Số quan trọng nhất là **nhúng 1 câu hỏi** — nó nằm trên đường găng của ngân sách
`< 2.5s tới byte audio đầu` (NT1).

Đọc số thế nào cho đúng:

```
Colab GPU (T4)      lạc quan  - máy chạy runtime không có T4 thì đừng chép số này vào spec
Colab CPU (2 vCPU)  bi quan   - máy 8 luồng ở nhà sẽ nhanh hơn
```

Muốn số **thật** cho spec §2 thì phải đo trên chính máy chạy runtime. Số ở đây chỉ để biết
thứ tự độ lớn và để so GPU với CPU.

In [ ]:
Q_DEMO = "lam sao luu bieu do ra file anh"


def bench(n=10, batch=1):
    encode([Q_DEMO])                                   # warmup
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    ts = []
    for _ in range(n):
        t0 = time.perf_counter()
        encode([Q_DEMO] * batch, batch=batch)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    ts.sort()
    return ts[len(ts) // 2], ts[int(len(ts) * 0.95) - 1]


med, p95 = bench()
print("[%s] nap model       %.1fs" % (DEVICE, LOAD_SEC))
print("[%s] nhung %d chunk   %.1fs" % (DEVICE, len(texts), EMBED_SEC))
print("[%s] nhung 1 cau hoi  %.0f ms (trung vi) | %.0f ms (p95)" % (DEVICE, med, p95))

In [ ]:
# TUY CHON - do lai tren CPU cua Colab (2 vCPU, bi quan hon may 8 luong).
# Chuyen model qua lai mat ~30s, bo qua duoc.
RUN_CPU_BENCH = False

if RUN_CPU_BENCH and DEVICE == "cuda":
    _dev = DEVICE
    model.cpu().float()
    if sparse_linear is not None:
        sparse_linear.cpu()
    DEVICE = "cpu"
    torch.set_num_threads(2)
    med_c, p95_c = bench(n=5)
    print("[cpu] nhung 1 cau hoi %.0f ms (trung vi) | %.0f ms (p95) | %d luong" %
            (med_c, p95_c, torch.get_num_threads()))
    DEVICE = _dev
    model.to(DEVICE)
    if USE_FP16:
        model.half()
    if sparse_linear is not None:
        sparse_linear.to(DEVICE)
    print("da tra model ve", DEVICE)

## 9. Thử truy vấn

`filter_divider=True` là **mặc định đúng cho runtime**: chunk trang phân mục chỉ có tên
chương lặp hai lần, không bị nội dung nào làm loãng nên điểm rất cao — robot nhảy vào
**trang trắng**, đúng loại `harmful_jump` mà §11 chặn dưới 2%.

Vẫn giữ cờ tắt được, để *đo* hiện tượng đó chứ không phải để giấu nó.

In [ ]:
def search(q, k=5, filter_divider=True, show=True):
    qv, _ = encode([q])
    pool = [i for i, c in enumerate(chunks)
            if not (filter_divider and c["content_type"] == "section_divider")]
    sims = MAT[pool] @ qv[0]
    order = np.argsort(-sims)[:k]
    hits = [(float(sims[j]), chunks[pool[j]]) for j in order]
    if show:
        print("? %s   (loc trang phan muc: %s)" % (q, filter_divider))
        for s, c in hits:
            print("  %.3f  tr%-3d %-6s %-16s %s" %
                    (s, c["page_no"], c["vector_role"], c["content_type"],
                     c["text_raw"][:64].replace("\n", " / ")))
        print()
    return hits


for q in ["lam sao luu bieu do ra file anh",
          "do thi dang duong",
          "ve ban do the gioi bang python"]:
    search(q)
    search(q, filter_divider=False)

### Hybrid dense + sparse (tạm)

Chuẩn hoá min-max rồi cộng có trọng số. Đây là **bản tạm để nhìn**, không phải cách trộn
cuối cùng — chỉnh trọng số là việc của `search.py` với bộ eval có nhãn, không phải đoán ở
đây (§6: ngưỡng phải fit trên eval, không hardcode).

In [ ]:
def _dot_sparse(a, b):
    if len(a) > len(b):
        a, b = b, a
    return sum(w * b[t] for t, w in a.items() if t in b)


def search_hybrid(q, k=5, w_sparse=0.3, filter_divider=True, show=True):
    if not SPARSE:
        return search(q, k, filter_divider, show)
    qv, qs = encode([q], want_sparse=True)
    pool = [i for i, c in enumerate(chunks)
            if not (filter_divider and c["content_type"] == "section_divider")]
    d = MAT[pool] @ qv[0]
    s = np.array([_dot_sparse(qs[0], SPARSE[i]) for i in pool], dtype=np.float32)
    nrm = lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
    total = (1 - w_sparse) * nrm(d) + w_sparse * nrm(s)
    order = np.argsort(-total)[:k]
    if show:
        print("? %s   (hybrid, w_sparse=%.2f)" % (q, w_sparse))
        for j in order:
            c = chunks[pool[j]]
            print("  tong %.3f (dense %.3f | sparse %.2f)  tr%-3d %s" %
                    (total[j], d[j], s[j], c["page_no"],
                     c["text_raw"][:56].replace("\n", " / ")))
        print()
    return [(float(total[j]), chunks[pool[j]]) for j in order]


search_hybrid("plt.savefig luu file png")
search("plt.savefig luu file png")

## 10. Self-retrieval audit — gate §11 ≥ 90%

Lấy **`text_raw`** của từng chunk làm truy vấn (bỏ tiền tố — dùng thẳng `text_enriched`
thì cosine = 1.0, phép thử thành vô nghĩa), index vẫn là `text_enriched`. Top-1 phải rơi
đúng **trang** của chunk đó.

Trượt nghĩa là hai trang **biểu diễn lẫn nhau** — R2 sẽ nhảy sai ở runtime. Phải biết
điều đó **trước khi** S4 tiêu tiền sinh kịch bản.

Ba con số phải đọc cùng nhau:

| Số | Nghĩa |
|---|---|
| `top1_rate` | gate §11, cần ≥ 0.90 |
| `margin` (top1 − top2) | biên càng mỏng, confidence gate của R2 càng hay phải hỏi lại |
| `top1_excl_self` | bỏ chính nó ra: trang đó còn ai đại diện được không |

In [ ]:
def audit(filter_divider):
    pool = [i for i, c in enumerate(chunks)
            if not (filter_divider and c["content_type"] == "section_divider")]
    P = MAT[pool]
    Q, _ = encode([chunks[i]["text_raw"] for i in pool])
    S = Q @ P.T

    ok, ok_hard, margins, fails = 0, 0, [], []
    for r, i in enumerate(pool):
        want = chunks[i]["page_no"]
        order = np.argsort(-S[r])
        top = [(float(S[r][j]), chunks[pool[j]]) for j in order[:3]]
        margins.append(top[0][0] - top[1][0] if len(top) > 1 else 1.0)
        if top[0][1]["page_no"] == want:
            ok += 1
        else:
            fails.append({
                "query_chunk": chunks[i]["chunk_id"], "want_page": want,
                "got": [{"chunk_id": c["chunk_id"], "page_no": c["page_no"],
                         "score": round(s, 4)} for s, c in top],
            })
        row = S[r].copy()
        row[r] = -1.0                                   # bo chinh no ra
        if chunks[pool[int(np.argmax(row))]]["page_no"] == want:
            ok_hard += 1

    n = len(pool)
    res = {
        "filter_divider": filter_divider,
        "n_query": n,
        "top1_rate": round(ok / n, 4),
        "top1_excl_self_rate": round(ok_hard / n, 4),
        "margin_median": round(float(np.median(margins)), 4),
        "margin_p10": round(float(np.percentile(margins, 10)), 4),
        "failures": fails,
    }
    print("--- loc trang phan muc: %s" % filter_divider)
    print("    top-1            %.1f%% (%d/%d)   gate §11 >= 90%% -> %s" %
            (res["top1_rate"] * 100, ok, n, "DAT" if res["top1_rate"] >= 0.90 else "TRUOT"))
    print("    top-1 bo self    %.1f%%" % (res["top1_excl_self_rate"] * 100))
    print("    margin           trung vi %.3f | p10 %.3f" %
            (res["margin_median"], res["margin_p10"]))
    for f in fails:
        print("    TRUOT %s (muon trang %d) -> %s" %
                (f["query_chunk"].split("#")[-1], f["want_page"],
                 ", ".join("tr%d:%.3f" % (g["page_no"], g["score"]) for g in f["got"])))
    print()
    return res


AUDIT = {"model": MODEL_ID, "doc_id": CS["doc_id"], "pooling": POOLING,
         "n_chunk": len(chunks), "device": DEVICE,
         "filtered": audit(True), "unfiltered": audit(False)}

(OUT / "audit" / "self_retrieval.json").write_text(
    json.dumps(AUDIT, ensure_ascii=False, indent=2), encoding="utf-8")
print("ghi -> out/kb/audit/self_retrieval.json")
print("chenh lech do loc trang phan muc: %+.1f diem %%" %
        ((AUDIT["filtered"]["top1_rate"] - AUDIT["unfiltered"]["top1_rate"]) * 100))

### Kiểm tra trang trùng tiêu đề

Câu hỏi của `kb-chunk.md §10`: mấy trang liền nhau cùng một tiêu đề, **tiền tố tách được
bao nhiêu?** Cosine giữa chúng càng sát 1.0 thì R2 càng dễ nhảy nhầm trong cùng một mục.

In [ ]:
from collections import defaultdict

g = defaultdict(list)
for i, c in enumerate(chunks):
    g[c.get("section_title") or "(khong co muc)"].append(i)

print("%-34s %3s  %-24s %s" % ("muc", "n", "cosine trong muc", "trang"))
for title, idxs in sorted(g.items(), key=lambda kv: -len(kv[1])):
    if len(idxs) < 2:
        continue
    M = MAT[idxs] @ MAT[idxs].T
    off = M[~np.eye(len(idxs), dtype=bool)]
    pages = sorted({chunks[i]["page_no"] for i in idxs})
    print("%-34s %3d  min %.3f  max %.3f      %s" %
            (title[:34], len(idxs), off.min(), off.max(),
             "%d-%d" % (pages[0], pages[-1]) if len(pages) > 1 else str(pages[0])))

## 11. Tải kết quả về

Giải nén vào `out/kb/` của repo. File `.npy` không đọc được bằng mắt — thứ tự hàng nằm ở
`rows` trong `.vectors.json`.

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/kb_vectors", "zip", OUT)
print("%.1f MB  %s" % (Path(zip_path).stat().st_size / 1e6, zip_path))
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  %8d KB  %s" % (p.stat().st_size // 1024, p.relative_to(OUT)))

from google.colab import files
files.download(zip_path)

### Tuỳ chọn — sinh sẵn cache cho `src/kb/embed.py`

`Embedder` ở máy cache theo `sha1(model_id + "\0" + text)`. Sinh sẵn mấy file `.npy` này
rồi bỏ vào `out/kb/.embed_cache/bge-m3/` thì lần chạy `embed_chunkset` ở máy **không phải
forward lại** chunk nào.

Nói cho thật: `embed_chunkset` vẫn **nạp model trước rồi mới đọc cache** — cache tiết kiệm
thời gian forward, *không* tránh được 2.2 GB tải model ở máy.
Chỉ dùng cache này khi `USE_FP16 = False` (fp16 lệch số so với bản chạy CPU).

In [ ]:
MAKE_LOCAL_CACHE = False

if MAKE_LOCAL_CACHE:
    import hashlib
    assert not USE_FP16, "fp16 lech so voi ban chay CPU -> khong dung lam cache"
    cdir = OUT / ".embed_cache" / MODEL_SLUG
    cdir.mkdir(parents=True, exist_ok=True)
    for t, v in zip(texts, MAT):
        key = hashlib.sha1(f"{MODEL_ID}\x00{t}".encode()).hexdigest()
        np.save(cdir / f"{key}.npy", v)
    print("ghi %d file cache -> %s" % (len(texts), cdir))
    print("chep ca thu muc .embed_cache vao out/kb/ cua repo")

## 12. Về lại repo

```bash
unzip kb_vectors.zip -d out/kb/
```

Kết quả rơi đúng chỗ `docs/spec/embedding.md §8` đã định:

```
out/kb/<doc_id>__bge-m3.vectors.npy
out/kb/<doc_id>__bge-m3.vectors.json
out/kb/<doc_id>__bge-m3.sparse.json
out/kb/audit/self_retrieval.json
```

Đọc lại ở máy:

```python
import json, numpy as np
mat  = np.load("out/kb/<doc_id>__bge-m3.vectors.npy")
meta = json.load(open("out/kb/<doc_id>__bge-m3.vectors.json", encoding="utf-8"))
row  = {cid: i for i, cid in enumerate(meta["rows"])}     # chunk_id -> hang
```

**Hai việc tiếp theo, theo thứ tự:**

1. Đọc `audit/self_retrieval.json`. `top1_rate < 0.90` là **trượt gate §11** — sửa chunk
   hoặc sửa tiền tố **trước**, không đi tiếp. Kết quả audit quyết định việc tiếp theo,
   không phải phỏng đoán (spec §10).
2. `src/kb/embed.py` đang log `sparse CHUA co -> moi la dense`. Notebook này sinh được
   sparse rồi; muốn khớp thì cổng sparse vào `embed.py`, hoặc ghi rõ ở đó là sparse đến từ
   đường Colab.